# Weekly Update Generator - AgentCore Deployment

## Overview

In this tutorial we will learn how to build and deploy an automated weekly status report generator using Amazon Bedrock AgentCore Runtime. The agent collects data from multiple sources (team updates, meeting notes, metrics, bug trackers), performs analysis, generates visualizations, and uploads comprehensive reports to S3.

### Architecture & Files

![Architecture Diagram](./images/architecture.png)

- **Agent Implementation**: `agent.py` - Main agent file that defines the Bedrock Agent Core application, agent configuration, and entrypoint
- **Tools**: `tools.py` - Contains 16 tools for data reading, analysis, visualization, and S3 upload
- **Demo Data**: `demo_data/` directory containing:
  - `project_status/` - CSV files with project progress and blockers
  - `team_updates/` - Markdown files with individual team member updates
  - `metrics/` - CSV files with KPI data (current and historical)
  - `issues/` - JSON files with bug tracker data
  - `meeting_notes/` - Markdown files with meeting summaries

### How It Works

1. **Data Collection** - The agent dynamically discovers and reads the latest week's data:
   - Scans directories for files matching patterns like `projects_week_XX.csv`, `kpis_week_XX.csv`, `bug_tracker_week_XX.json`
   - Automatically uses the most recent week number found
   - Reads all team member markdown files and meeting notes

2. **Data Analysis** - The agent performs intelligent analysis:
   - Validates data quality and completeness across all sources
   - Cross-references information (e.g., matching project names in status files vs team updates)
   - Performs sentiment analysis on team updates to detect morale issues
   - Calculates risk scores based on project health, blockers, and bug severity

3. **Visualization Generation** - Creates PNG charts using matplotlib:
   - Bug severity distribution (pie/bar charts)
   - KPI metrics trends with historical comparison
   - Project timeline and progress visualization
   - Team velocity charts
   - Predictive forecast models for key metrics

4. **Report Synthesis** - Compiles a structured markdown report with:
   - Executive summary with key highlights and concerns
   - Detailed sections for projects, team updates, KPIs, bugs, and meetings
   - Risk analysis and blockers
   - Action items and next week priorities

5. **S3 Upload** - Automatically uploads to S3:
   - Final markdown report
   - All generated chart images
   - Organized by date in the S3 bucket for easy retrieval

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Asynchronous agent                                                       |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 4.5                                                        |
| Tutorial components | Multi-tool agent, data analysis, visualization, S3 integration, AgentCore Runtime|
| Tutorial vertical   | Business Operations & Reporting                                                  |
| Example complexity  | Intermediate                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK, boto3, matplotlib, scikit-learn              |

### Tutorial Architecture

This tutorial demonstrates how to deploy an asynchronous reporting agent to AgentCore runtime. The agent orchestrates 16 different tools to create comprehensive weekly status reports automatically.

### Tutorial Key Features

* Hosting an asychronous, multi-tool agent on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models (Claude Sonnet 4)
* Using Strands Agents SDK
* S3 integration for data storage and report delivery

### Deployment

The agent runs as a Bedrock Agent Core application that can be:
- Invoked locally for testing
- Deployed to AWS Lambda for production use
- Called via API with async task tracking (returns immediately while processing in background)
- Monitored via ping endpoint that reports HEALTHY or HEALTHY_BUSY status


## Prerequisites
- AWS Account with access to Amazon Bedrock AgentCore
- AWS credentials
- Python 3.12+
- S3 bucket for storing demo data and reports

## Setup and Imports

In [1]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import time
import json
import boto3
import re
from datetime import datetime, timedelta
from pathlib import Path
from botocore.exceptions import ClientError

print("✅ Imports successful!")

/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✅ Imports successful!


## Pre-deployment Configuration

In [2]:
boto_session = Session()
region = boto_session.region_name
agent_name = "weekly_update_generator"

# TODO: Replace with your S3 bucket name
S3_BUCKET = "a-sample-dataset-6"
S3_PREFIX = "demo_data"

print(f"📍 Region: {region}")
print(f"🤖 Agent name: {agent_name}")
print(f"🪣 S3 Bucket: {S3_BUCKET}")

📍 Region: us-east-1
🤖 Agent name: weekly_update_generator
🪣 S3 Bucket: a-sample-dataset-6


## Step 1: Update Demo Data and Upload to S3

In [3]:
# Run the update script
!python update_demo_dates.py --bucket {S3_BUCKET} --prefix {S3_PREFIX}


📅 Updating demo data to current week:
   Week 9 of 2026
   Week of February 23, 2026

Renaming files...

Updating file contents...
✓ Updated kpis_historical.csv
✓ Updated kpis_week_09.csv
✓ Updated projects_week_09.csv
✓ Updated sprint_planning_feb_23.md
✓ Updated incident_postmortem_feb_23.md
✓ Updated executive_sync_feb_23.md
✓ Updated bug_tracker_week_09.json
✓ Updated carlos_mendez_week_09.md
✓ Updated jordan_lee_week_09.md
✓ Updated samantha_brooks_week_09.md
✓ Updated maya_thompson_week_09.md
✓ Updated alex_rivera_week_09.md

✅ Demo data updated successfully!
   All dates now reflect week 9 (February 23 - February 23, 2026)

📝 Updating tools.py configuration...
✓ Updated S3_BUCKET in tools.py to: a-sample-dataset-6

Uploading to S3 bucket: a-sample-dataset-6
   Prefix: demo_data/

✓ Uploaded demo_data/metrics/kpis_historical.csv
✓ Uploaded demo_data/metrics/kpis_week_09.csv
✓ Uploaded demo_data/project_status/projects_week_09.csv
✓ Uploaded demo_data/meeting_notes/sprint_plannin

## Step 2: Deploy Agent to AgentCore

The CreateAgentRuntime operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent.

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment
First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Tools are packaged with the agent so that the agent can access them at runtime.

![](images/configure.png)

In [4]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)

print("✅ Agent configured")
response


Entrypoint parsed: file=/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/agent.py, bedrock_agentcore_name=agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: weekly_update_generator


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-asy
nc-agents/Dockerfile

Generated .dockerignore: /Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/.dockerignore
Setting 'weekly_update_generator' as default agent
Bedrock AgentCore configured: /Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/.bedrock_agentcore.yaml


✅ Agent configured


ConfigureResult(config_path=PosixPath('/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/Dockerfile'), dockerignore_path=PosixPath('/Users/nadhyap/Repos/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/04-async-agents/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='213160203681', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

### Launch the agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

![](images/launch.png)

In [5]:
launch_result = agentcore_runtime.launch()

print("🚀 Agent launched")
print(f"Agent ARN: {launch_result.agent_arn}")
launch_result

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'weekly_update_generator' to account 213160203681 (us-east-1)
Generated image tag: 20260225-130326-073
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: weekly_update_generator
ECR repository available: 213160203681.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-weekly_update_generator
Getting or creating execution role for agent: weekly_update_generator
Using AWS region: us-east-1, account ID: 213160203681
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-710c5be322


✅ Reusing existing ECR repository: 213160203681.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-weekly_update_generator


✅ Reusing existing execution role: arn:aws:iam::213160203681:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-710c5be322
Execution role available: arn:aws:iam::213160203681:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-710c5be322
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: weekly_update_generator
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-710c5be322
Reusing existing CodeBuild execution role: arn:aws:iam::213160203681:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-710c5be322
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: weekly_update_generator/source.zip
Updated CodeBuild project: bedrock-agentcore-weekly_update_generator-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.7s
🔄 DOWNLOAD_SOURCE started

🚀 Agent launched
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:213160203681:runtime/weekly_update_generator-P9Z8FrFodP


LaunchResult(mode='codebuild', tag='bedrock_agentcore-weekly_update_generator:None', env_vars=None, port=None, runtime=None, ecr_uri='213160203681.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-weekly_update_generator:20260225-130326-073', agent_id='weekly_update_generator-P9Z8FrFodP', agent_arn='arn:aws:bedrock-agentcore:us-east-1:213160203681:runtime/weekly_update_generator-P9Z8FrFodP', codebuild_id='bedrock-agentcore-weekly_update_generator-builder:465536d5-11f6-4467-95ba-36bfb6fc7cdd', build_output=None)

## Step 3: Wait for Agent to be Ready

Let's check the agent's deployment status to ensure it can be invoked succesfully

In [6]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Status: {status}")

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"\n✅ Agent is {status}")

Retrieved Bedrock AgentCore status for: weekly_update_generator


Status: READY

✅ Agent is READY


## Step 4: Add S3 Permissions to Execution Role

For this tutorial, we will read and write data stored in an Amazon S3 bucket. Now that our agent has been launched, we need to give the agent permission to access the S3 bucket.

In [7]:
# Get execution role from agent runtime
agent_runtime_id = launch_result.agent_arn.split('/')[-1]
print(f"Agent Runtime ID: {agent_runtime_id}")

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)


response = agentcore_client.get_agent_runtime(
    agentRuntimeId=agent_runtime_id,
    agentRuntimeVersion='1'
)

execution_role_arn = response.get('roleArn')

execution_role_name = execution_role_arn.split('/')[-1]
print(f"✓ Execution role: {execution_role_name}")

iam_client = boto3.client('iam')
policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
        "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"]
    }]
}

iam_client.put_role_policy(
    RoleName=execution_role_name,
    PolicyName='WeeklyReportsS3Access',
    PolicyDocument=json.dumps(policy_document)
)

print(f"\n✅ S3 permissions added!")
print(f"   Role: {execution_role_name}")
print(f"   Bucket: {S3_BUCKET}")
    


Agent Runtime ID: weekly_update_generator-P9Z8FrFodP
✓ Execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-710c5be322

✅ S3 permissions added!
   Role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-710c5be322
   Bucket: a-sample-dataset-6


## Step 5: Invoke the Agent

Finally, we can invoke our agent and have it generate our weekly report

In [8]:
from datetime import datetime, timedelta
import time
import json

# Get current week info
today = datetime.now()
monday = today - timedelta(days=today.weekday())

print(f"🚀 Starting async agent invocation for week of {monday.strftime('%B %d, %Y')}...\n")

# Invoke the agent
start_time = time.time()
invoke_response = agentcore_runtime.invoke({
    "prompt": f"Generate the weekly status report for the week of {monday.strftime('%B %d, %Y')}."
})

print(f"✅ Agent invocation started")

# Extract task info from response
if 'response' in invoke_response and invoke_response['response']:
    task_info = json.loads(invoke_response['response'][0])
    task_id = task_info.get('task_id')
    print(f"📋 Task ID: {task_id}")
    print(f"📊 Initial Status: {task_info.get('status')}")
    print(f"💬 {task_info.get('message')}\n")

print(f"⏳ Polling agent status...\n")

poll_count = 0
time.sleep(5)  # Give agent time to start

while True:
    poll_count += 1
    elapsed = time.time() - start_time
    
    try:
        # Ping the agent to determine busy status"
        ping_response = agentcore_runtime.invoke({
            "method": "ping",
            "payload": {}
        })
        
        if 'response' in ping_response and ping_response['response']:
            response_data = json.loads(ping_response['response'][0])
            health_status = response_data.get('status', 'Unknown')
            active_tasks = response_data.get('active_tasks', 0)
            
            print(f"Poll #{poll_count} ({elapsed:.1f}s): {health_status} (active tasks: {active_tasks})")
            
            # Check if agent is Healthy (no active tasks)
            if health_status == 'Healthy':
                print(f"\n✅ Task completed in {elapsed:.1f} seconds")
                break
                
    except Exception as e:
        print(f"Poll #{poll_count} ({elapsed:.1f}s): Error - {e}")
    
    if elapsed > 600:
        print(f"\n⚠️ Timeout after {elapsed:.1f} seconds")
        break
    
    time.sleep(5)

# Printing report output path
week_num = today.isocalendar()[1]
year = today.year
week_folder = f"{year}/week_{week_num:02d}_{monday.strftime('%Y-%m-%d')}"
print(f"\n📁 Report should be at: s3://{S3_BUCKET}/weekly_reports/{week_folder}/weekly_report.md")


🚀 Starting async agent invocation for week of February 23, 2026...

✅ Agent invocation started
📋 Task ID: -6658798990924175355
📊 Initial Status: BUSY
💬 Weekly report generation started (Task ID: -6658798990924175355). Agent status is now BUSY.

⏳ Polling agent status...

Poll #1 (22.2s): HealthyBusy (active tasks: 1)
Poll #2 (30.9s): HealthyBusy (active tasks: 1)
Poll #3 (37.9s): HealthyBusy (active tasks: 1)
Poll #4 (43.8s): HealthyBusy (active tasks: 1)
Poll #5 (51.3s): HealthyBusy (active tasks: 1)
Poll #6 (57.0s): Healthy (active tasks: 0)

✅ Task completed in 57.0 seconds

📁 Report should be at: s3://a-sample-dataset-6/weekly_reports/2026/week_09_2026-02-23/weekly_report.md


## Parse and Display Agent Response

## Cleanup (Optional)
Let's now clean up the AgentCore Runtime created

In [9]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)
print("Deleting Runtime")

In [10]:
repo_name = launch_result.ecr_uri.split('/')[-1].split(':')[0]

ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)
response = ecr_client.delete_repository(
    repositoryName=repo_name,
    force=True
)
print("Deleting ECR")